# A reproducible baseline pipeline

**Task:** rank 2026 vulnerabilities (CVEs) so that the top 500 of 72,670 holds as many as possible that CISA later adds to its Known Exploited Vulnerabilities list (KEV).

**What this notebook delivers**, one requirement at a time:

| # | Requirement | Where |
|---|---|---|
| 1 | Train / validation split | §2: time-based, 2024 → 2025 |
| 2 | Missing values | §3-4: median / most-frequent imputation, fitted on training rows only |
| 3 | Categorical encoding | §4: one-hot for CVSS parts; point-in-time history rates for the assigner |
| 4 | Scaling | §4: standardization for the linear model only |
| 5 | Simple baseline | §5: random ranking, and the benchmark's "exploit code then EPSS" rule |
| 6 | At least three algorithms | §5: logistic regression, random forest, histogram gradient boosting, LightGBM |
| 7 | Suitable metrics | §6: hits in the top 0.69%, precision/recall at that cut, average precision, ROC AUC |
| 8 | Training and validation performance | §7 |
| 9 | Leakage checks | §8: six automated checks |
| 10 | Reproducibility | §1 and §9: fixed seeds, sorted rows, recorded versions, identical re-run |

**No hyperparameter tuning.** Every model uses library defaults, except for handling class imbalance and fixing the seed. These are reference results to improve on later.

**The code** lives in the `ml/` package. This notebook calls it step by step and explains each choice. Run everything at once with `python -m ml.run` from the repo root, which writes `outputs/ml_baselines/`.

| Module | Job |
|---|---|
| `ml/config.py` | Seed, paths, split windows, feature lists, banned columns |
| `ml/data.py` | Loads the CSVs and builds the labeled row sets |
| `ml/featureset.py` | Point-in-time features (reuses the team's `scripts/features*.py`) |
| `ml/models.py` | Preprocessing (`ColumnTransformer`) and the candidate `Pipeline`s |
| `ml/evaluate.py` | Ranking metrics, bootstrap ranges and paired comparisons |
| `ml/leakage.py` | Automated leakage checks |
| `ml/run.py` | Ties it together and saves the results |

## 1. Reproducibility setup

- **One seed (42)** goes to Python, NumPy and every model's `random_state`.
- **Rows are always sorted by `cve_id`.** Databases don't promise a stable row order, and both tie-breaking and bootstrapping depend on it.
- **Library versions are recorded** with every saved result.

In [1]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = next(p for p in [Path(".."), Path(".")] if (p / "ml" / "config.py").exists()).resolve()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
from ml import config, data, evaluate, leakage, models
from ml.featureset import feature_matrix
from ml.run import set_seeds, build_datasets, evaluate_models
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 20)

set_seeds(config.SEED)
import sklearn, platform
print(f"seed {config.SEED} | Python {platform.python_version()} | scikit-learn {sklearn.__version__} | "
      f"NumPy {np.__version__} | pandas {pd.__version__}")

seed 42 | Python 3.9.6 | scikit-learn 1.6.1 | NumPy 2.0.2 | pandas 2.3.3


## 2. The train / validation split: by time, not at random

| | Rows | Label |
|---|---|---|
| **Train** | CVEs published January-September **2024** | added to KEV after the publish month and by **Dec 31, 2024** |
| **Validation** | CVEs published January-September **2025** | added to KEV after the publish month and by **Dec 31, 2025** |

**Why by time:** the real task predicts the *future* from the *past*. A random split would put 2025 rows in training and leak later knowledge into the model: for example, which vendors were exploited later in 2025. Training ends, labels included, before validation starts, exactly as it will for 2026.

**Why these windows:** they copy the official benchmark's rules (January-September publications, rows already on KEV removed, signals from the publish month). Phase 3 checked that this backtest reproduces the benchmark's best standard ranking.

In [2]:
train_rows, valid_rows, train, valid, score = build_datasets()
k_tr, k_va = evaluate.k_for(len(train)), evaluate.k_for(len(valid))
pd.DataFrame({"rows": [len(train), len(valid), len(score)],
              "positives": [int(train.y.sum()), int(valid.y.sum()), None],
              "positive rate": [f"{train.y.mean():.2%}", f"{valid.y.mean():.2%}", "unknown"],
              "top-k evaluated": [k_tr, k_va, 500]}, index=["train (2024)", "validation (2025)", "2026 scoreboard"])

,rows,positives,positive rate,top-k evaluated
train (2024),28839,33.0,0.11%,198
validation (2025),35281,52.0,0.15%,243
2026 scoreboard,72670,NaN,unknown,500


**Reading this:** positives are rare (about 1 in 700 to 1 in 900), so every result is noisy. That's why §6 uses ranking metrics, and §7 adds bootstrap ranges.

## 3. Features, and why missing values need care here

All features come from `ml/featureset.py`, which reuses the team's leak-checked feature code. Every *history* feature (an assigner's, product's or weakness type's past KEV rate) counts only KEV entries dated **on or before the end of the row's snapshot month**.

| Type | Count | Examples |
|---|---|---|
| Numeric | 7 | EPSS percentile in its month, CVSS score, assigner KEV rate, advisory batch size |
| Binary | 25 | Exploit-code flags, EPSS missing, description keywords, weakness-type groups |
| Categorical | 5 | CVSS parts: attack vector, complexity, privileges, user interaction, scope |

**What's missing, and why it matters:**

In [3]:
X_tr = feature_matrix(train)[config.FEATURES]
miss = pd.DataFrame({"train missing": X_tr.isna().mean(), "validation missing": feature_matrix(valid)[config.FEATURES].isna().mean(),
                     "2026 missing": feature_matrix(score)[config.FEATURES].isna().mean()})
miss = miss[miss.max(axis=1) > 0].sort_values("2026 missing", ascending=False)
miss.style.format("{:.1%}")

,train missing,validation missing,2026 missing
cvss3_score,0.4%,3.8%,8.7%
attack_vector,0.4%,3.8%,8.7%
attack_complexity,0.4%,3.8%,8.7%
privileges_required,0.4%,3.8%,8.7%
user_interaction,0.4%,3.8%,8.7%
scope,0.4%,3.8%,8.7%


**Reading this:** only the six CVSS columns have gaps, and the gap **grows**: 0.4% of training rows, 3.8% of validation rows and 8.7% of 2026 rows. NIST hasn't reviewed most new CVEs yet. That decides how to handle them:

- **Impute, but never flag the blank.** In the past data, a blank CVSS score mostly means "NIST hasn't reviewed it", and NIST reviews exploited CVEs first. A "score is missing" column would therefore partly encode the answer: a leak that would look great in backtests and fail on 2026, where blanks are far more common. So blanks are filled with typical values: the training **median** for numbers, the **most frequent** category for categories.
- **The imputation values are learned from training rows only** (inside the `Pipeline`), so validation and 2026 data never influence them.
- `signal_missing` (no EPSS that month) *is* kept as a flag. It reflects the EPSS feed's coverage, not NIST's review.

## 4. Preprocessing: one `ColumnTransformer`, fitted on training rows only

| Column type | Step 1 | Step 2 | Why |
|---|---|---|---|
| Numeric (7) | median imputation | standardize (linear model only) | The median ignores extreme values; logistic regression needs comparable scales, trees don't |
| Binary (25) | most-frequent imputation | standardize (linear model only) | Already 0/1 |
| Categorical (5) | most-frequent imputation | one-hot encode | Few categories each; `handle_unknown="ignore"` so a new category in 2026 can't crash scoring |

**The assigner (who issued the CVE) is not one-hot encoded.** There are hundreds of assigners, many new ones appear every year, and a one-hot model would memorize the few seen with positives. It's represented instead by its **point-in-time KEV rate** (`log_assigner_kev_rate`): a target-history encoding computed only from KEV entries before each row's snapshot, so it's leak-free and works for unseen assigners.

In [4]:
prep = models.make_preprocessor(scale=True).fit(X_tr, train.y)
names = prep.get_feature_names_out()
print(f"{len(config.FEATURES)} input columns -> {len(names)} model inputs after one-hot encoding")
print("one-hot columns:", [n for n in names if any(n.startswith(c + "_") for c in config.CATEGORICAL)])
num_imp = prep.named_transformers_["num"].named_steps["impute"]
print("medians learned from TRAINING rows only:",
      {c: round(float(v), 3) for c, v in zip(config.NUMERIC, num_imp.statistics_)})

37 input columns -> 45 model inputs after one-hot encoding
one-hot columns: ['attack_vector_ADJACENT_NETWORK', 'attack_vector_LOCAL', 'attack_vector_NETWORK', 'attack_vector_PHYSICAL', 'attack_complexity_HIGH', 'attack_complexity_LOW', 'privileges_required_HIGH', 'privileges_required_LOW', 'privileges_required_NONE', 'user_interaction_NONE', 'user_interaction_REQUIRED', 'scope_CHANGED', 'scope_UNCHANGED']
medians learned from TRAINING rows only: {'epss_pct': 0.429, 'cvss3_score': 6.7, 'log_assigner_kev_rate': -7.545, 'log_assigner_rate_12m': -7.111, 'log_batch_size': 3.135, 'log_cwe_kev_rate': -6.183, 'log_product_kev_prior': 0.0}


## 5. The models

Each model is a `Pipeline` (preprocessing + estimator), so preprocessing is always refit together with the model, on training rows only.

| Model | Why it's here | Non-default settings |
|---|---|---|
| **Random ranking** (`DummyClassifier`) | The floor: what chance alone scores | seed |
| **Baseline: exploit code then EPSS** | The benchmark's best standard ranking, needing no learning. **The bar to beat** | none |
| **Logistic regression** | Simple, linear, hard to overfit, and its weights are readable | `class_weight="balanced"`, scaled inputs |
| **Random forest** | Many deep trees averaged; captures interactions without scaling | `class_weight="balanced_subsample"` |
| **Histogram gradient boosting** | scikit-learn's boosted trees: strong on tabular data | `class_weight="balanced"` |
| **LightGBM** | The industry-standard boosted trees, used in the team model | `class_weight="balanced"`; deterministic single-thread mode |

**Class imbalance:** with about 1 positive per 700 rows, an unweighted model can lower its loss by predicting "no" everywhere. Balanced class weights make each positive count as much as all the negatives together. That changes *what* the model learns from, not its capacity, so it isn't tuning.

**Everything else is a library default.** No tree depth, learning rate or regularization was chosen. These are baselines.

In [5]:
for name, pipe in models.make_models().items():
    print(f"{name:34} -> {type(pipe.named_steps['model']).__name__}")

Random ranking (DummyClassifier)   -> DummyClassifier
Baseline: exploit code then EPSS   -> ExploitThenEPSS
Logistic regression                -> LogisticRegression
Random forest                      -> RandomForestClassifier
Histogram gradient boosting        -> HistGradientBoostingClassifier
LightGBM                           -> LGBMClassifier


## 6. How models are scored

**Accuracy is useless here:** answering "never exploited" for every row is 99.85% accurate and finds nothing. The real task is a **ranking**: a security team patches the top of the list, and the judges count real positives in the top 500. So:

| Metric | Question it answers |
|---|---|
| **hits@k** (main) | How many later-exploited CVEs are in the top 0.69%? That's 243 rows in validation and 198 in training, the benchmark's share |
| precision@k | What fraction of that short list is a real positive? |
| recall@k | What fraction of all positives made the list? |
| average precision | How good is the ranking overall, weighted toward the top? It suits rare positives |
| ROC AUC | How often a random positive ranks above a random negative (0.5 = chance). Secondary: it can hide poor performance at the top |
| **90% bootstrap range** | How much hits@k could move from the luck of which CVEs happen to be in the sample |
| **vs. baseline** | Paired bootstrap: a lead is real only if its range excludes 0 |

## 7. Results: training vs. validation

In [6]:
results, val_scores = evaluate_models(train, valid)
results.round(3)

,fit_seconds,train hits@k,train AUC,train AP,valid hits@k,valid 90% range,valid precision@k,valid recall@k,valid AUC,valid AP,valid vs. baseline
model,,,,,,,,,,,
Random ranking (DummyClassifier),0.1,0,0.500,0.001,0,0-0,0.000,0.000,0.500,0.001,-14 (-20 to -9)
Baseline: exploit code then EPSS,0.0,7,0.625,0.026,14,9-20,0.058,0.269,0.768,0.047,
Logistic regression,0.3,13,0.991,0.066,11,6-16,0.045,0.212,0.844,0.048,-3 (-10 to +2)
Random forest,0.6,33,1.000,0.924,12,7-18,0.049,0.231,0.708,0.025,-2 (-9 to +4)
Histogram gradient boosting,0.4,30,0.997,0.355,11,5-16,0.045,0.212,0.653,0.017,-3 (-10 to +1)
LightGBM,0.3,33,1.000,0.989,11,6-16,0.045,0.212,0.841,0.036,-3 (-9 to +3)


**Reading this:**
- **The floor and the bar:** random ranking finds 0, and the exploit-then-EPSS rule finds **14 of 52** in the top 243 (90% range 9-20).
- **No default model beats the rule yet.** Logistic regression finds 11, random forest 12, and both boosted-tree models 11. Every paired range against the baseline includes 0, so none is clearly worse either.
- **Clear overfitting in the tree models:** random forest and LightGBM find **all 33** training positives in the training top 198 (training AUC 1.0), then 11-12 on validation. Histogram gradient boosting is close behind (30 of 33). With only 33 positives, default trees grow until they memorize them.
- **Logistic regression ranks best overall** (validation AUC 0.84, with LightGBM's 0.84 alongside), and its training and validation hits are closest (13 vs. 11). It still fits the training set closely (training AUC 0.99), and it doesn't separate the very top better than the rule.
- **What this means for the next step:** the gap is about *regularization*, not about features. The team's tuned model (`demo_findings_and_model.ipynb`), with stronger regularization, shallow trees and an evidence-chosen feature set, reaches **18** on this same split. Tuning should be judged against these baseline numbers.

## 8. Leakage checks

| Check | What it would catch |
|---|---|
| Banned / "is blank" columns | NIST review status, reference counts, or blank flags reaching a model |
| Time order | Any training row or label dated on or after the start of validation |
| No overlap | The same CVE in two sets |
| Exclusion rule | A row that was already on KEV at its snapshot (the label would be known) |
| **Planted future KEV** | Feature code reading the future: 3,000 fake KEV entries dated 2030 are added, and the features are rebuilt. Nothing may change |
| Single-feature screen | One input that separates the classes almost perfectly on its own, a classic leak signature |

In [7]:
leaks = leakage.run_all(train_rows, valid_rows, train, valid, score)
leaks

,check,passed,detail
0,No banned or 'is blank' columns are model inputs,True,banned found: none; blank flags: none
1,Training rows and labels all end before valida...,True,"train published to 2024-09-30, labels to 2024-..."
2,No CVE appears in more than one set,True,shared IDs: 0
3,No row was already on KEV at its snapshot,True,rows on KEV by month end: 0
4,"3,000 fake KEV entries dated 2030 change no fe...",True,changed: none
5,No single feature has AUC above 0.9 on its own,True,highest: log_assigner_kev_rate = 0.875


**Reading this:** all six checks pass. The strongest single feature is the assigner's past KEV rate (AUC 0.875 on its own). That's high but plausible, since vendors with exploited history keep getting exploited, and it's built only from KEV entries before each snapshot, as the planted-future test confirms.

## 9. Reproducibility check: run it again

In [8]:
again, _ = evaluate_models(train, valid)
same = results.drop(columns="fit_seconds").equals(again.drop(columns="fit_seconds"))
print("Second run identical to the first (all metrics, all models):", same)
assert same

Second run identical to the first (all metrics, all models): True


## Summary

- **The pipeline:** a time-based split, point-in-time features, leak-aware imputation, one-hot encoding, scaling only where it matters, and six models in scikit-learn `Pipeline`s. It's fully reproducible (an identical re-run is verified) and passes six leakage checks.
- **The baseline numbers to improve on (2025 validation, top 243 of 52):** the exploit-then-EPSS rule scores **14**. Default models score 11-12, and the tree models memorize their training positives.
- **Next:** tune regularization (tree depth, minimum leaf size, the logistic regression's `C`) with cross-validation on the training years only, and compare against these baselines with the same paired test.

`python -m ml.run` reproduces everything and writes `outputs/ml_baselines/results.csv`, `leakage_checks.csv` and `run_info.json`.